# bitladder: KWS experiments in Colab

Runtime: GPU (T4 is enough). All logic lives in the repository scripts; this notebook only
clones the repo, fetches the feature cache and runs a chain of experiments.

Data options (run **one** of the two cells in section 2):

* **2a** download a prebuilt `kws_cache.tar` shared by link (no Drive mount, nothing else on
  your Drive is exposed);
* **2b** download Speech Commands v2 from tensorflow.org and rebuild the cache (~5 min).

Section 3 holds one chain per day; run the one you need. Section 4 collects the JSON files.

In [ ]:
# 1. Code
import os
REPO = 'https://github.com/Vzzzz/bitladder.git'
if not os.path.exists('/content/bitladder'):
    !git clone -q {REPO} /content/bitladder
%cd /content/bitladder
!git pull -q --ff-only
!pip install -q -e . 2>&1 | tail -1
import torch; print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0))
!python -m pytest -q 2>&1 | tail -2

In [ ]:
# 2a. Prebuilt feature cache from a link-shared file (Google Drive file id or any direct URL).
# Build it locally once: tar -cf kws_cache.tar -C data kws_cache
CACHE_FILE_ID = '1Ho0bzfCwmfxS65FYsntta4HbjkAcQSkP'   # kws_cache.tar on Google Drive (link-shared); or set CACHE_URL
CACHE_URL = ''
import os
os.makedirs('data', exist_ok=True)
if CACHE_FILE_ID:
    !pip install -q gdown && gdown -q {CACHE_FILE_ID} -O /content/kws_cache.tar
elif CACHE_URL:
    !wget -q -O /content/kws_cache.tar "{CACHE_URL}"
!tar -xf /content/kws_cache.tar -C data && ls -la data/kws_cache

In [ ]:
# 2b. Rebuild the cache from the public dataset (CC BY 4.0), ~5 min on a T4.
!mkdir -p data/speech_commands_v2 data/speech_commands_test_v2
!wget -q -O /content/sc.tar.gz http://download.tensorflow.org/data/speech_commands_v0.02.tar.gz
!wget -q -O /content/sct.tar.gz http://download.tensorflow.org/data/speech_commands_test_set_v0.02.tar.gz
!tar -xzf /content/sc.tar.gz -C data/speech_commands_v2 && tar -xzf /content/sct.tar.gz -C data/speech_commands_test_v2
!python audio/kws/cache.py --root data/speech_commands_v2 --test-root data/speech_commands_test_v2 --out data/kws_cache 2>&1 | grep -v '^  '
!tar -cf /content/kws_cache.tar -C data kws_cache && ls -la /content/kws_cache.tar  # download this once and share it for 2a

In [ ]:
# 3a. (2026-10-23) Branch bits at k=8 / k=16 on a 4-bit base, pair {stop, go}, 3 seeds. Done; kept for reference.
B = '--keywords yes,no,up,down,left,right,on,off --exclude stop,go --unknown-ratio 1.0'
!python audio/kws/run.py --epochs 30 --name kws12_fp32_s0 | grep ^RESULT
for s in (0, 1, 2):
    !python audio/kws/run.py {B} --epochs 30 --seed {s} --name base8_unk1_fp32_s{s} | grep ^RESULT
    !python audio/kws/run.py {B} --epochs 15 --lr 1e-3 --scheme uniform --wbits 4 --abits 8 --init results/base8_unk1_fp32_s{s}.pt --seed {s} --name base8_unk1_u4a8_s{s} | grep ^RESULT
    for k in (8, 16):
        for cfg, a in [('fp32', ''), ('u8a8', '--scheme uniform --wbits 8 --abits 8 --no-first-last'),
                       ('u4a8', '--scheme uniform --wbits 4 --abits 8 --no-first-last'),
                       ('t158a8', '--scheme ternary --abits 8 --no-first-last')]:
            !python audio/kws/expand.py --base results/base8_unk1_u4a8_s{s}.json --new stop,go --epochs 20 --k {k} --seed {s} {a} --name exp_k{k}_base-u4a8_branch-{cfg}_s{s} | grep ^RESULT

In [ ]:
# 3b. (2026-10-24) Second pair {on, off} + depthwise-separable branch (k=16 -> 7.1k params, inside the 10k cap), 3 seeds.
# Both pairs get a fresh 4-bit base (the 23rd's session is gone). ~60 min on a T4.
PAIRS = {'stopgo': ('yes,no,up,down,left,right,on,off', 'stop,go'),
         'onoff':  ('yes,no,up,down,left,right,stop,go', 'on,off')}
BRANCHES = [('fp32', ''), ('u4a8', '--scheme uniform --wbits 4 --abits 8 --no-first-last'),
            ('t158a8', '--scheme ternary --abits 8 --no-first-last')]
for pair, (base_kw, new) in PAIRS.items():
    B = f'--keywords {base_kw} --exclude {new} --unknown-ratio 1.0'
    for s in (0, 1, 2):
        !python audio/kws/run.py --progress {B} --epochs 30 --seed {s} --name base8_{pair}_fp32_s{s} | grep ^RESULT
        !python audio/kws/run.py --progress {B} --epochs 15 --lr 1e-3 --scheme uniform --wbits 4 --abits 8 --init results/base8_{pair}_fp32_s{s}.pt --seed {s} --name base8_{pair}_u4a8_s{s} | grep ^RESULT
        for arch, extra in [('k16ds', '--k 16 --separable'), ('k8', '--k 8'), ('k16', '--k 16')]:
            for cfg, a in BRANCHES:
                !python audio/kws/expand.py --progress --base results/base8_{pair}_u4a8_s{s}.json --new {new} --epochs 20 {extra} --seed {s} {a} --name exp_{pair}_{arch}_base-u4a8_branch-{cfg}_s{s} | grep ^RESULT

In [ ]:
# 3c. (2026-10-24) Layer analysis of the 12-class model: weight / activation stats and one-shot sensitivity heatmap.
!python audio/kws/run.py --progress --epochs 30 --name kws12_fp32_s0 | grep ^RESULT
!python audio/kws/analyze.py --run results/kws12_fp32_s0.json --split validation --batches 10 2>&1 | tail -40
from IPython.display import Image, display
display(Image('results/kws12_fp32_s0_sensitivity.png'))

In [ ]:
# 4. Collect a table and pack results for download (JSON, md, png; checkpoints stay here).
import json, glob, pandas as pd
rows = []
for f in sorted(glob.glob('results/exp_*.json')):
    r = json.load(open(f))
    rows.append({'name': r['name'], 'new': '-'.join(r['new_words']), 'k': r['k'], 'separable': r.get('separable', False),
                 'seed': r['seed'], 'branch': r['config']['scheme'], 'wbits': r['config']['wbits'],
                 'branch_params': r['branch_params'], 'branch_bytes': r['branch_bytes'],
                 'frr_ungated': r['test_macro_frr_ungated'], 'frr_soft': r.get('test_macro_frr_soft'),
                 'frr_hard': r['test_macro_frr'], 'core_fires_on_new': r['core_fires_on_new']})
df = pd.DataFrame(rows)
if len(df):
    display(df.groupby(['new', 'k', 'separable', 'branch', 'wbits'])['frr_ungated'].agg(['mean', 'std', 'count']))
    df.to_csv('results/summary.csv', index=False)
!tar -czf /content/results_json.tgz results/*.json results/*.md results/*.png results/summary.csv 2>/dev/null; ls -la /content/results_json.tgz
from google.colab import files; files.download('/content/results_json.tgz')